In [ ]:
import numpy as np
import jax.numpy as jnp
from hair_module import (
    interpolate_emmisivity_vmap,
    detect_noisy_bands,
    destripe,
    denoise,
    correct_band,
    lookup_B_vector_batch,
    get_sky_pixel,
)

from hadar_module_v2 import (
    load_data,
    process_sky_spectrum,
    generate_planck_lut,
    plot_T_map,
    plot_e_map,
    plot_bright_map,
)


def main_stage_1(
    file_path,
    file_name,
    row_mean_threshold,
    band_problem_ratio,
    max_bad_band_ratio,
):

    hsi_noisy, hsi_wav, wav_unit = load_data(
        filepath=file_path,
        hsi_file_name=file_name,
        value2radiance=1,
        HSI_mat_dict_name="hsi",
    )

    noisy_band_indices, stripe_strengths = detect_noisy_bands(
        hsi_noisy,
        row_mean_threshold,
        band_problem_ratio,
        max_bad_band_ratio,
    )

    all_band_indices = np.arange(len(hsi_wav))
    good_band_indices = np.setdiff1d(all_band_indices, noisy_band_indices).astype(int)

    _, hsi_destriped_good = destripe(
        jnp.asarray(stripe_strengths),
        jnp.asarray(hsi_noisy[..., noisy_band_indices]),
        lambda_TVx=0.006,
        lambda_TVy=0.013,
    )

    hsi_denoised_good = denoise(hsi_destriped_good)

    return (
        hsi_wav,
        wav_unit,
        hsi_noisy,
        hsi_denoised_good,
        good_band_indices,
        all_band_indices,
    )


def main_stage_2(hsi_wav, wav_unit, good_band_indices, hsi_noisy_good_denoised, sky):
    band_cor = True
    hsi_wav_good = hsi_wav[good_band_indices]
    if band_cor:
        hsi_wav_good_cor, simulated_sky, c0_opt, c1_opt, c2_opt = correct_band(
            sky, hsi_wav_good, "highres_sky_reference.txt"
        )
    else:
        hsi_wav_good_cor = hsi_wav_good

    e_opt = None
    T_opt = None
    X_opt = None
    bright_opt = None
    config = None

    # Due to privilege restrictions, Code about HADAR is not provided in this release.

    if band_cor:
        e_cor_opt = interpolate_emmisivity_vmap(e_opt, hsi_wav_good_cor, hsi_wav_good)

    e_full = interpolate_emmisivity_vmap(e_cor_opt, hsi_wav_good, hsi_wav)
    complete_B_lut, _ = generate_planck_lut(hsi_wav, config, wav_unit)
    B = lookup_B_vector_batch(T_opt, complete_B_lut, config)
    term_1 = e_full * B
    term_2 = (1 - e_full) * X_opt
    HSI_recovered = term_1 + term_2
    H, W, _ = hsi_noisy_good_denoised.shape
    e_full = e_full.reshape(H, W, -1)
    X_opt = X_opt.reshape(H, W, -1)
    T_opt = T_opt.reshape(H, W)
    bright_opt = bright_opt.reshape(H, W)
    HSI_recovered = HSI_recovered.reshape(H, W, -1)

    return e_full, bright_opt, T_opt, HSI_recovered

In [ ]:
file_input_folder = "./data"
file_name = "1"

(
    hsi_wav,
    wav_unit,
    hsi_noisy,
    hsi_denoised_good,
    good_band_indices,
    all_band_indices,
) = main_stage_1(
    file_input_folder,
    file_name,
    row_mean_threshold=1.005,
    band_problem_ratio=0.1,
    max_bad_band_ratio=0.7,
)

In [ ]:
sky, _ = process_sky_spectrum(hsi_denoised_good, get_sky_pixel(file_name))

In [ ]:
T_opt, e_opt, X_opt, bright_opt, hsi = main_stage_2(
    hsi_wav,
    wav_unit,
    good_band_indices,
    hsi_denoised_good,
    sky,
)

In [ ]:
fig_T_opt = plot_T_map(T_opt, base_height=400)
display(fig_T_opt)

fig_e_opt = plot_e_map(e_opt, hsi_wav, band_idx=110, base_height=300)
display(fig_e_opt)

fig_bright_opt = plot_bright_map(bright_opt, cmap="gray", base_height=300)
display(fig_bright_opt)